## Traitement données pédologiques
### Structure générale

1. Traitement des couches en entrée, notamment:
    a. Pondération des couches données par pronfondeur (sable, limon, argile, humus) par la profondeur utile (PU) Si la PU est plus faible que la profondeur donnée d'une couche, alors sa pondération est affaiblie jusqu'à zéro
    b. Rasterisation des couches vecteurs
    c. Normalisation
2. Harmonisation de couches d'un même paramètre, mais avec des classifications différentes (pierrosité, mouillure)
3. Agrégation des couches en donnant priorité aux couches les plus précises (ici merge_priorite_gratier)
4. Normalisation de toutes les couches avec un clip percentiles des extrêmes afin de "nettoyer" les effets de bord.
5. Calcul de l'indice de Sol

### 1. Traitement des couches en entrée

1.a Pondération des couches données par pronfondeur (sable, limon, argile, humus) par la profondeur utile (PU) Si la PU est plus faible que la profondeur donnée d'une couche, alors sa pondération est affaiblie jusqu'à zéro

In [ ]:
# Pondération des couches par PU (profondeur utile)
import rioxarray as rio
import xarray as xr
import numpy as np

def weighted_by_root_depth(h0_30_path, h30_60_path, h60_120_path, pu_path, output_path):
    """
    Combine three soil horizon rasters (0–30, 30–60, 60–120 cm)
    using depth-weighted averaging based on useful rooting depth (PU).

    Parameters
    ----------
    h0_30_path : str   path to raster for 0–30 cm horizon
    h30_60_path : str  path to raster for 30–60 cm horizon
    h60_120_path : str path to raster for 60–120 cm horizon
    pu_path : str      path to raster of useful rooting depth (cm)
    output_path : str  path for output raster
    """

    # --- Load rasters ---
    h1 = rio.open_rasterio(h0_30_path).squeeze()
    h2 = rio.open_rasterio(h30_60_path).squeeze()
    h3 = rio.open_rasterio(h60_120_path).squeeze()
    PU = rio.open_rasterio(pu_path).squeeze()

    # --- Sanity check: ensure PU has no negative depth ---
    PU = xr.where(PU < 0, 0, PU)

    # --- Compute weights for each horizon ---
    w1 = xr.where(PU > 0,  np.minimum(PU, 30), 0)
    w2 = xr.where(PU > 30, np.minimum(PU - 30, 30), 0)
    w3 = xr.where(PU > 60, np.minimum(PU - 60, 60), 0)

    # --- Total weight (avoid division by zero) ---
    Wtot = w1 + w2 + w3
    Wtot = xr.where(Wtot == 0, 1, Wtot)

    # --- Depth-weighted mean ---
    Vfinal = (h1 * w1 + h2 * w2 + h3 * w3) / Wtot

    # --- Write CRS and export ---
    Vfinal = Vfinal.rio.write_crs(h1.rio.crs)
    Vfinal.rio.to_raster(output_path)

    print(f"Weighted raster saved to {output_path}")



In [ ]:
# sable
weighted_by_root_depth(
    r"..\data\raw\KOBO\Geodaten_Sand_CH\Geodaten_Sand_CH\CH_Sand_0_30\CH_Sand_0_30.tif",
    r"..\data\raw\KOBO\Geodaten_Sand_CH\Geodaten_Sand_CH\CH_Sand_30_60\CH_Sand_30_60.tif",
    r"..\data\raw\KOBO\Geodaten_Sand_CH\Geodaten_Sand_CH\CH_Sand_60_120\CH_Sand_60_120.tif",
    r"..\data\raw\sols\prof_utile.tif",
    "../data/processed/Sol/sable_weighted.tif"
)
# limon
weighted_by_root_depth(
    r"..\data\raw\KOBO\Geodaten_Schluff_CH\Geodaten_Schluff_CH\CH_Schluff_0_30\CH_Schluff_0_30.tif",
    r"..\data\raw\KOBO\Geodaten_Schluff_CH\Geodaten_Schluff_CH\CH_Schluff_30_60\CH_Schluff_30_60.tif",
    r"..\data\raw\KOBO\Geodaten_Schluff_CH\Geodaten_Schluff_CH\CH_Schluff_60_120\CH_Schluff_60_120.tif",
    r"..\data\raw\sols\prof_utile.tif",
    "../data/processed/Sol/limon_weighted.tif"
)
# argile
weighted_by_root_depth(
    r"..\data\raw\KOBO\Geodaten_Ton_CH\Geodaten_Ton_CH\CH_Ton_0_30\CH_Ton_0_30.tif",
    r"..\data\raw\KOBO\Geodaten_Ton_CH\Geodaten_Ton_CH\CH_Ton_30_60\CH_Ton_30_60.tif",
    r"..\data\raw\KOBO\Geodaten_Ton_CH\Geodaten_Ton_CH\CH_Ton_60_120\CH_Ton_60_120.tif",
    r"..\data\raw\sols\prof_utile.tif",
    "../data/processed/Sol/argile_weighted.tif"
)
# Humus
weighted_by_root_depth(
    r"..\data\raw\KOBO\Geodaten_Humus_CH\Geodaten_Humus_CH\CH_Humus_0_30\CH_Humus_0_30.tif",
    r"..\data\raw\KOBO\Geodaten_Humus_CH\Geodaten_Humus_CH\CH_Humus_30_60\CH_Humus_30_60.tif",
    r"..\data\raw\KOBO\Geodaten_Humus_CH\Geodaten_Humus_CH\CH_Humus_60_120\CH_Humus_60_120.tif",
    r"..\data\raw\sols\prof_utile.tif",
    "../data/processed/Sol/humus_weighted.tif"
)

1.b Rasterization des couches vecteurs

In [ ]:
# Rasterization des cartes vecteurs (Gratier GEO.vd et BLW)
# Indispensable pour l'intégration au pipeline raster

import geopandas as gpd
import rasterio
from rasterio import features
import numpy as np

def rasterize_vector(input_vector, attribute, reference_raster, output_raster):
    """
    Rasterise un attribut d'un fichier vectoriel en se basant sur un raster de référence.
    """

    # ---- Charger le vecteur ----
    gdf = gpd.read_file(input_vector)
    gdf = gdf.to_crs(rasterio.open(reference_raster).crs)

    # ---- Charger le raster de référence ----
    with rasterio.open(reference_raster) as ref:
        meta = ref.meta.copy()
        transform = ref.transform
        shape = (ref.height, ref.width)

    # ---- Préparer les géométries ----
    shapes = [(geom, value) for geom, value 
              in zip(gdf.geometry, gdf[attribute]) 
              if geom is not None]

    # ---- Rasterisation ----
    rasterized = features.rasterize(
        shapes=shapes,
        out_shape=shape,
        transform=transform,
        fill=-9999,         # NoData
        dtype='float32'
    )

    # ---- Enregistrement ----
    meta.update({"dtype": "float32", "nodata": -9999})
    with rasterio.open(output_raster, "w", **meta) as dst:
        dst.write(rasterized, 1)

    print(f"Raster saved: {output_raster}")


In [ ]:
# rasterisation Gratier regime hydrique
rasterize_vector(
    input_vector=r"..\data\raw\sols\DGE_sols_clean.shp",
    attribute="REGIME_HYD",
    reference_raster=r"..\data\raw\sols\prof_utile.tif",
    output_raster=r"..\data\processed\Sol\Gratier_Regime_Hydrique.tif"
) 
# rasterisation Gratier pierrosité
rasterize_vector(
    input_vector=r"..\data\raw\sols\DGE_sols_clean.shp",
    attribute="PIERROSITE",
    reference_raster=r"..\data\raw\sols\prof_utile.tif",
    output_raster=r"..\data\processed\Sol\Gratier_Pierrosite.tif"
)

# rasterization blw pierrosité
rasterize_vector(   
    input_vector=r"..\data\raw\sols\BLW_sols_vaud.shp",
    attribute="SKELETT",
    reference_raster=r"..\data\raw\sols\prof_utile.tif",
    output_raster=r"..\data\processed\Sol\BLW_Skelett.tif"
)

# rasterization blw capacité en rétention d'eau (wasserpsei)
rasterize_vector(   
    input_vector=r"..\data\raw\sols\BLW_sols_vaud.shp",
    attribute="WASSERSPEI",
    reference_raster=r"..\data\raw\sols\prof_utile.tif",
    output_raster=r"..\data\processed\Sol\BLW_Wasserspei.tif"
)

# rasterization blw perméabilité (wasserdurchlässigkeit)
rasterize_vector(   
    input_vector=r"..\data\raw\sols\BLW_sols_vaud.shp",
    attribute="WASSERDURC",
    reference_raster=r"..\data\raw\sols\prof_utile.tif",
    output_raster=r"..\data\processed\Sol\BLW_Wasserdurchlaessigkeit.tif"
)

#rastization blw mouillure (vernass)
rasterize_vector(   
    input_vector=r"..\data\raw\sols\BLW_sols_vaud.shp",
    attribute="VERNASS",
    reference_raster=r"..\data\raw\sols\prof_utile.tif",
    output_raster=r"..\data\processed\Sol\BLW_Vernass.tif"
)

1.c Normalisation

In [ ]:
# normalization avec forçage entre 0 et 1
import rasterio


def normalize_raster(input_raster, output_raster, nodata_val=-9999):
    """
    Normalise un raster sur [0,1], en conservant les NoData.

    Parameters
    ----------
    input_raster : str
        Chemin vers le raster à normaliser.
    output_raster : str
        Chemin vers le raster normalisé.
    nodata_val : float
        Valeur de NoData dans le raster.
    """
    
    with rasterio.open(input_raster) as src:
        data = src.read(1).astype('float32')
        profile = src.profile

        # Mask des NoData
        mask = (data == nodata_val)

        # Normalisation en ignorant les NoData
        valid = data[~mask]
        min_val = valid.min()
        max_val = valid.max()
        data[~mask] = (valid - min_val) / (max_val - min_val)

        # Remettre les NoData
        data[mask] = nodata_val

        # Mise à jour du profil
        profile.update(dtype='float32', nodata=nodata_val)

        # Sauvegarde
        with rasterio.open(output_raster, 'w', **profile) as dst:
            dst.write(data, 1)

    print(f"Raster normalisé sauvegardé dans : {output_raster}")




In [ ]:
# normalisation des rasters de sol
normalize_raster(
    input_raster=r"..\data\processed\Sol\argile_weighted.tif",
    output_raster=r"..\data\processed\Sol\argile_weighted_normalized.tif"
)

normalize_raster(
    input_raster=r"..\data\processed\Sol\BLW_Skelett.tif",
    output_raster=r"..\data\processed\Sol\BLW_Skelett_normalized.tif"
)
normalize_raster(
    input_raster=r"..\data\processed\Sol\BLW_Vernass.tif",
    output_raster=r"..\data\processed\Sol\BLW_Vernass_normalized.tif"
)
normalize_raster(
    input_raster=r"..\data\processed\Sol\BLW_Wasserdurchlaessigkeit.tif",
    output_raster=r"..\data\processed\Sol\BLW_Wasserdurchlaessigkeit_normalized.tif"
)
normalize_raster(
    input_raster=r"..\data\processed\Sol\BLW_Wasserspei.tif",
    output_raster=r"..\data\processed\Sol\BLW_Wasserspei_normalized.tif"
)
normalize_raster(
    input_raster=r"..\data\processed\Sol\Gratier_Pierrosite.tif",
    output_raster=r"..\data\processed\Sol\Gratier_Pierrosite_normalized.tif"
)
normalize_raster(
    input_raster=r"..\data\processed\Sol\Gratier_Regime_Hydrique.tif",
    output_raster=r"..\data\processed\Sol\Gratier_Regime_Hydrique_normalized.tif"
)
normalize_raster(
    input_raster=r"..\data\processed\Sol\argile_weighted.tif",
    output_raster=r"..\data\processed\Sol\argile_weighted_normalized.tif"
)
normalize_raster(
    input_raster=r"..\data\processed\Sol\limon_weighted.tif",
    output_raster=r"..\data\processed\Sol\limon_weighted_normalized.tif"
)
normalize_raster(
    input_raster=r"..\data\processed\Sol\sable_weighted.tif",
    output_raster=r"..\data\processed\Sol\sable_weighted_normalized.tif"
)
normalize_raster(
    input_raster=r"..\data\processed\Sol\humus_weighted.tif",
    output_raster=r"..\data\processed\Sol\humus_weighted_normalized.tif"
)
normalize_raster(
    input_raster=r"..\data\raw\sols\prof_utile.tif",
    output_raster=r"..\data\processed\Sol\prof_utile_normalized.tif"
)


In [ ]:
# Normalisation avancée avec clipping percentile et options
import os
import json
import numpy as np
import rasterio
from rasterio.enums import Resampling

def normalize_with_clip(input_path, output_path, *,
                        pmin=2.0, pmax=98.0,
                        prelog=False,
                        symmetric=False,
                        nodata_keep=True,
                        nodata_value=None,
                        overwrite=False):
    """
    Normalise un raster avec clipping percentile et min-max.
    - symmetric=True : pour variables centrées sur 0 (ex: courbure). On calcule le percentile pmax
      de la distribution des valeurs absolues puis on clip à +/- that_value.
    - prelog=True : applique np.log1p avant clipping (utile pour TWI)
    - pmin,pmax : percentiles (0..100) ; if symmetric only pmax used.
    - nodata_keep : conserve nodata (remplacé par np.nan pendant calcul).
    """
    if os.path.exists(output_path) and not overwrite:
        raise FileExistsError(f"{output_path} exists. Use overwrite=True to replace.")

    with rasterio.open(input_path) as src:
        profile = src.profile.copy()
        arr = src.read(1).astype(np.float32)
        src_nodata = src.nodata if nodata_value is None else nodata_value

    # mask nodata
    if src_nodata is not None:
        mask = (arr == src_nodata) | (~np.isfinite(arr))
    else:
        mask = ~np.isfinite(arr)
    valid = arr[~mask]

    if valid.size == 0:
        raise ValueError("No valid pixels found in input raster.")

    # optionally apply pre-log (on valid values only)
    if prelog:
        valid_proc = np.log1p(valid)
    else:
        valid_proc = valid

    # symmetric clipping around 0 for centered variables
    if symmetric:
        # use percentile on absolute values
        abs_vals = np.abs(valid_proc)
        cutoff = np.percentile(abs_vals, pmax)
        low_val = -cutoff
        high_val = cutoff
        print(f"[symmetric] cutoff (abs) @ p{pmax} = {cutoff:.6g}")
    else:
        low_val = np.percentile(valid_proc, pmin)
        high_val = np.percentile(valid_proc, pmax)
        print(f"percentiles: p{pmin}={low_val:.6g}, p{pmax}={high_val:.6g}")

    # Create working copy (float) and replace nodata with nan
    work = arr.astype(np.float32)
    work[mask] = np.nan

    # if prelog, transform entire working array's valid pixels
    if prelog:
        with np.errstate(invalid='ignore'):
            work = np.where(np.isfinite(work), np.log1p(work), work)

    # clip
    work_clipped = np.clip(work, low_val, high_val)

    # min-max normalize
    denom = (high_val - low_val)
    if denom == 0:
        # constant field after clipping
        norm = np.full_like(work_clipped, np.nan, dtype=np.float32)
        norm[np.isfinite(work_clipped)] = 0.5
    else:
        norm = (work_clipped - low_val) / denom
    norm[mask] = profile.get("nodata", np.nan) if nodata_keep else np.nan

    # update profile
    profile.update(dtype=rasterio.float32, compress='deflate')
    if nodata_keep:
        profile.update(nodata=profile.get("nodata", -9999))
        out_nodata = profile["nodata"]
    else:
        out_nodata = np.nan

    # ensure output dir
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # write raster
    with rasterio.open(output_path, "w", **profile) as dst:
        write_arr = np.where(np.isfinite(norm), norm, out_nodata).astype(np.float32)
        dst.write(write_arr, 1)

    # save metadata about normalization
    meta = {
        "input": os.path.basename(input_path),
        "output": os.path.basename(output_path),
        "pmin": pmin,
        "pmax": pmax,
        "prelog": bool(prelog),
        "symmetric": bool(symmetric),
        "low_val": float(low_val),
        "high_val": float(high_val),
        "nodata_kept": bool(nodata_keep),
    }
    meta_path = os.path.splitext(output_path)[0] + "_meta.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"Saved normalized raster: {output_path}")
    print(f"Saved metadata: {meta_path}")
    return output_path, meta_path


### 2. Harmonisation de couches d'un même paramètre, mais avec des classifications différentes (pierrosité, mouillure)

In [ ]:
# Agrégration des classes de prierrosité
import rasterio
import numpy as np

def harmonise_pierrosite(in_raster, out_raster, source="gratier"):

    with rasterio.open(in_raster) as src:
        arr = src.read(1)
        profile = src.profile.copy()
        nodata = src.nodata if src.nodata is not None else -9999

    # dictionnaires d’agrégation
    if source.lower() == "gratier":
        mapping = {
            0: 0,
            1: 1,
            2: 2,
            3: 2,
            4: 3,
            5: 3,
            7: 4,
            9: 4,      # <-- blocs fusionnés dans la classe 4
            10: nodata,
            11: nodata
        }
    elif source.lower() == "blw":
        mapping = {
            -9999: nodata,
            0: nodata,
            1: 0,
            2: 1,
            3: 2,
            4: 3,
            5: 4
        }
    else:
        raise ValueError("source must be 'gratier' or 'blw'")

    # reclassification
    out_arr = np.full(arr.shape, nodata, dtype=np.float32)

    for old_val, new_val in mapping.items():
        out_arr[arr == old_val] = new_val

    # écriture
    profile.update(dtype=rasterio.float32, nodata=nodata)
    with rasterio.open(out_raster, "w", **profile) as dst:
        dst.write(out_arr, 1)

    print(f"Harmonisation terminée pour {source}: {out_raster}")


In [ ]:
harmonise_pierrosite(
    in_raster=r"..\data\processed\Sol\Gratier_Pierrosite.tif",
    out_raster=r"..\data\processed\Sol\Gratier_Pierrosite_harmonized.tif",
    source="gratier"
)
harmonise_pierrosite(
    in_raster=r"..\data\processed\Sol\BLW_Skelett.tif",
    out_raster=r"..\data\processed\Sol\BLW_Skelett_harmonized.tif",
    source="blw"
)

In [ ]:
# Agrégration des classes de mouillure
import rasterio
import numpy as np

def harmonise_mouillure(in_raster, out_raster, source="gratier"):

    with rasterio.open(in_raster) as src:
        arr = src.read(1)
        profile = src.profile.copy()
        nodata = src.nodata if src.nodata is not None else -9999

    # dictionnaires d’agrégation
    if source.lower() == "gratier":
        mapping = {
            1: 0,   # sans excès d'eau
            2: 1,   # hydromorphie faible
            3: 2,   # modérément engorgés
            4: 2,   # humides, rarement engorgés
            5: 3,   # très humide, souvent engorgé
            6: 4,   # mouillé dès la surface
            7: -9999,  # non sondé
            8: -9999,  # non déterminé
        }
    elif source.lower() == "blw":
        mapping = {
            -9999: -9999,
            0: -9999,   # inconnu
            1: 0,       # pas de mouillure
            2: 1,       # humide
            3: 2,       # faiblement mouillé
            4: 3        # mouillé
        }
    else:
        raise ValueError("source must be 'gratier' or 'blw'")

    # reclassification
    out_arr = np.full(arr.shape, nodata, dtype=np.float32)

    for old_val, new_val in mapping.items():
        out_arr[arr == old_val] = new_val

    # écriture
    profile.update(dtype=rasterio.float32, nodata=nodata)
    with rasterio.open(out_raster, "w", **profile) as dst:
        dst.write(out_arr, 1)

    print(f"Harmonisation terminée pour {source}: {out_raster}")


In [ ]:
harmonise_mouillure(
    in_raster=r"..\data\processed\Sol\Gratier_Regime_Hydrique.tif",
    out_raster=r"..\data\processed\Sol\Gratier_Regime_Hydrique_harmonized.tif",
    source="gratier"
)
harmonise_mouillure(
    in_raster=r"..\data\processed\Sol\BLW_Vernass.tif",
    out_raster=r"..\data\processed\Sol\BLW_Vernass_harmonized.tif",
    source="blw"
)

### 3. Agrégation des couches en donnant priorité aux couches les plus précises (Gratier)

In [ ]:
# Merge des couches Gratier et BLW avec priorité à Gratier
import rasterio
import numpy as np

def merge_priorite_gratier(gratier_raster, blw_raster, output_raster):
    """
    Fusionne deux rasters de pierrosité harmonisés :
    - priorité à Gratier lorsque la valeur existe
    - utilisation de BLW dans les lacunes (NoData) de Gratier
    """

    # ---- Charger les deux rasters ----
    with rasterio.open(gratier_raster) as gsrc:
        g = gsrc.read(1)
        profile = gsrc.profile.copy()
        nodata_g = gsrc.nodata

    with rasterio.open(blw_raster) as bsrc:
        b = bsrc.read(1)
        nodata_b = bsrc.nodata

        # On peut vérifier la compatibilité spatiale :
        if (bsrc.height != gsrc.height) or (bsrc.width != gsrc.width):
            raise ValueError("Erreur : les deux rasters n'ont pas les mêmes dimensions.")

        if bsrc.transform != gsrc.transform:
            raise ValueError("Erreur : les rasters n'ont pas le même transform. "
                             "Ils doivent être reprojetés/alignés avant fusion.")

    # ---- Fusion ----
    merged = np.where(g != nodata_g, g, b)  # Gratier prioritaire

    # Si BLW a aussi du nodata, on garde nodata
    merged[(g == nodata_g) & (b == nodata_b)] = nodata_g

    # ---- Sauvegarde ----
    profile.update(dtype=rasterio.float32, nodata=nodata_g)

    with rasterio.open(output_raster, "w", **profile) as dst:
        dst.write(merged.astype(np.float32), 1)

    print(f"Raster fusionné sauvegardé : {output_raster}")


In [ ]:
merge_priorite_gratier(
    gratier_raster=r"..\data\processed\Sol\Gratier_Pierrosite_harmonized.tif",
    blw_raster=r"..\data\processed\Sol\BLW_Skelett_harmonized.tif",
    output_raster=r"..\data\processed\Sol\Pierrosite_merged.tif"
)

merge_priorite_gratier(
    gratier_raster=r"..\data\processed\Sol\Gratier_Regime_Hydrique_harmonized.tif",
    blw_raster=r"..\data\processed\Sol\BLW_Vernass_harmonized.tif",
    output_raster=r"..\data\processed\Sol\Mouillure_merged.tif"
)

normalize_with_clip(
    input_path=r"..\data\processed\Sol\Pierrosite_merged.tif",
    output_path=r"..\data\processed\Sol\Pierrosite_merged_normalized.tif",
    overwrite=True
)

### 4. Normalisation de toutes les couches avec un clip percentiles des extrêmes afin de "nettoyer" les effets de bord.

In [ ]:
normalize_with_clip(
    input_path=r"..\data\processed\Sol\Mouillure_merged.tif",
    output_path=r"..\data\processed\Sol\Mouillure_merged_normalized.tif",
)
normalize_with_clip(
    input_path=r"..\data\processed\Sol\argile_weighted.tif",
    output_path=r"..\data\processed\Sol\argile_weighted_normalized.tif"
)
normalize_with_clip(
    input_path=r"..\data\processed\Sol\limon_weighted.tif",
    output_path=r"..\data\processed\Sol\limon_weighted_normalized.tif"
)
normalize_with_clip(
    input_path=r"..\data\processed\Sol\sable_weighted.tif",
    output_path=r"..\data\processed\Sol\sable_weighted_normalized.tif"
)
normalize_with_clip(
    input_path=r"..\data\processed\Sol\humus_weighted.tif",
    output_path=r"..\data\processed\Sol\humus_weighted_normalized.tif"
)
normalize_with_clip(
    input_path=r"..\data\raw\sols\prof_utile.tif",
    output_path=r"..\data\processed\Sol\prof_utile_normalized.tif"
)
normalize_with_clip(input_path=r"..\data\processed\Sol\BLW_Skelett.tif",
                    output_path=r"..\data\processed\Sol\BLW_Skelett_normalized.tif",
                    pmin=2.0,pmax=100, overwrite=True)
normalize_with_clip(input_path=r"..\data\processed\Sol\BLW_Skelett.tif",
                    output_path=r"..\data\processed\Sol\BLW_Skelett_normalizedv2.tif",
                    pmin=0.0,pmax=100, overwrite=True)
normalize_with_clip(input_path=r"..\data\processed\Sol\Gratier_Pierrosite_cleaned.tif",
                    output_path=r"..\data\processed\Sol\Gratier_Pierrosite_cleaned_normalized.tif",
                    pmin=0.0,pmax=100, overwrite=True)
normalize_with_clip(input_path=r"..\data\processed\Sol\Gratier_Regime_Hydrique_cleaned.tif",
                    output_path=r"..\data\processed\Sol\Gratier_Regime_Hydrique_cleaned_normalized.tif",
                    pmin=0.0,pmax=100, overwrite=True)
normalize_with_clip(input_path=r"..\data\processed\Sol\BLW_Wasserspei.tif",
                    output_path=r"..\data\processed\Sol\BLW_Wasserspei_normalized.tif",
                    pmin=0.0,pmax=100, overwrite=True)
normalize_with_clip(input_path=r"..\data\processed\Sol\BLW_Wasserdurchlaessigkeit.tif",
                    output_path=r"..\data\processed\Sol\BLW_Wasserdurchlaessigkeit_normalized.tif",
                    pmin=0.0,pmax=100, overwrite=True)
normalize_with_clip(input_path=r"..\data\processed\Sol\BLW_Vernass.tif",
                    output_path=r"..\data\processed\Sol\BLW_vernass_normalized.tif",
                    pmin=0.0,pmax=100, overwrite=True)

### 5. Calcul de l'indice de sol

In [ ]:
# Calcul indice sol

import numpy as np
import rasterio
from tqdm import tqdm

# -------------------------
# 1. PARAMÈTRES UTILISATEUR
# -------------------------

# chemins vers les rasters (déjà normalisés et clipés)
rasters = {
    "prof":      r"../data/processed/Sol/prof_utile_normalized.tif",
    "argile":        r"../data/processed/Sol/argile_weighted_normalized.tif",
    "limon":  r"../data/processed/Sol/limon_weighted_normalized.tif",
    "sable":   r"../data/processed/Sol/sable_weighted_normalized.tif",
    "humus":     r"../data/processed/Sol/humus_weighted_normalized.tif",
    "retention":     r"../data/processed/Sol/BLW_Wasserspei_normalized.tif",
    "permeabilite":    r"../data/processed/Sol/BLW_Wasserdurchlaessigkeit_normalized.tif",
    "pierrosite":    r"../data/processed/Sol/Pierrosite_merged_normalized.tif",
    "mouillure":    r"../data/processed/Sol/Mouillure_merged_normalized.tif"
}

# pondérations (normalisées automatiquement)
weights = {
    "prof":      0.25,
    "argile":    0.10,
    "limon":     0.05,
    "sable":     0.05,
    "humus":     0.10,
    "retention":   0.05,
    "permeabilite":  0.10,
    "pierrosite":  0.15,
    "mouillure":    0.15
}

# variables à inverser hydrologiquement (haut = sec)
invert = {
    "prof":     True,
    "argile":       True,
    "limon": True,
    "sable":  False,
    "humus":    True,
    "retention":    True,
    "permeabilite":   False,
    "pierrosite":   False,
    "mouillure":   True,
}

output_path = r"../data/processed/Sol_index.tif"

# -------------------------
# 2. LIRE LES RASTERS
# -------------------------
arrays = {}
profiles = {}

for key, path in rasters.items():
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan
        arrays[key] = arr
        profiles[key] = src.profile

# -------------------------
# 3. RÉÉCHANTILLONNER / ALIGNER LES RASTERS
# -------------------------
from rasterio.warp import reproject, Resampling

# choisir un raster de référence (le premier)
ref_key = next(iter(arrays.keys()))
ref_profile = profiles[ref_key]
ref_transform = ref_profile["transform"]
ref_crs = ref_profile["crs"]
ref_shape = arrays[ref_key].shape

aligned_arrays = {}

for key, arr in arrays.items():
    dst = np.zeros(ref_shape, dtype=np.float32)

    reproject(
        source=arr,
        destination=dst,
        src_transform=profiles[key]["transform"],
        src_crs=profiles[key]["crs"],
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        resampling=Resampling.bilinear  # bilinear OK car déjà normalisé
    )

    aligned_arrays[key] = dst

arrays = aligned_arrays
# -------------------------
# 4. INVERSION DES RASTERS
# -------------------------
for key in arrays:
    if invert.get(key, False):
        arrays[key] = 1 - arrays[key]

# -------------------------
# 5. CALCUL DE L'INDICE TOPO
# -------------------------
weight_vec = np.array([weights[k] for k in arrays.keys()], dtype=float)
weight_vec = weight_vec / np.sum(weight_vec)

index = np.zeros(ref_shape, dtype=np.float32)

for i, key in enumerate(tqdm(arrays.keys(), desc="Combinaison")):
    index += arrays[key] * weight_vec[i]

# -------------------------
# 6. EXPORT
# -------------------------
profile_out = profiles[next(iter(arrays.keys()))].copy()
profile_out.update(dtype=rasterio.float32, count=1, compress="deflate", nodata=-9999)

with rasterio.open(output_path, "w", **profile_out) as dst:
    dst.write(np.where(np.isnan(index), -9999, index).astype(np.float32), 1)

print(f"✅ Indice topographique sauvegardé : {output_path}")


In [ ]:
# renormalisation de l'indice topo final
# il y a besoin, car moyenne pondérée ne sera jamais entre 0 et 1.
# cela permet d'augmenter le contraste final.
normalize_with_clip(input_path=r"../data/processed/Sol_index.tif",
                    output_path=r"../data/processed/Sol_index_normalized.tif",
                    pmin=0.1,pmax=99.9, overwrite=True)